# 09 — Tabular fusion: build table → clean → train → SHAP/ablation

Run scripts/fusion to build the fusion table and train the decision fusion model.


In [6]:

from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first) from REPO_ROOT."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check, cwd=REPO_ROOT)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = REPO_ROOT / c
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

fusion_scripts = [
  "scripts/fusion/build_fusion_training_table.py",
  "scripts/fusion/clean_fusion_table.py",
  "scripts/fusion/fit_decision_fusion.py",
  "scripts/fusion/shap_and_ablation.py",
  "scripts/fusion/feature_importance.py",
  "scripts/fusion/apply_tabular_model.py",
]
for s in fusion_scripts:
    p = REPO_ROOT / s
    print(s, "->", "OK" if p.exists() else "MISSING")


REPO_ROOT: /Users/ameerfiras/REDNET-ML
scripts/fusion/build_fusion_training_table.py -> OK
scripts/fusion/clean_fusion_table.py -> OK
scripts/fusion/fit_decision_fusion.py -> OK
scripts/fusion/shap_and_ablation.py -> OK
scripts/fusion/feature_importance.py -> OK
scripts/fusion/apply_tabular_model.py -> OK


## 9.1 Print help (keeps notebook synced)


In [11]:

for s in [
    "build_fusion_training_table.py",
    "clean_fusion_table.py",
    "fit_decision_fusion.py",
]:
    p = REPO_ROOT / "scripts/fusion" / s
    if p.exists():
        sh(f'python "{p}" --help', check=False)
    else:
        print("Missing:", p)



▶ python "/Users/ameerfiras/REDNET-ML/scripts/fusion/build_fusion_training_table.py" --help
[ok] plant_1079022886_hab.csv: rows=596
[ok] plant_1236881046_hab.csv: rows=632
[ok] plant_386838289_hab.csv: rows=633
[ok] plant_449632054_hab.csv: rows=398

✓ saved: runs/fusion/fusion_training_table_clean_populated.csv  rows=2259 cols=22

[coverage]
  p_tab: non-null 0.0%
  p_frcnn_r50_med: non-null 100.0%
  p_frcnn_mb_med: non-null 100.0%
  p_ssd_mb_med: non-null 100.0%
  sst: non-null 100.0%
  chlor_a: non-null 100.0%
  kd490: non-null 100.0%
  nflh: non-null 100.0%

▶ python "/Users/ameerfiras/REDNET-ML/scripts/fusion/clean_fusion_table.py" --help
✅ Cleaned and saved to runs/fusion/training_tables_draft/fusion_training_table_clean.csv  (197 rows kept)
   columns: ['tile', 'scene_id', 'datetime', 'month_key', 'fai_mean', 'rednir_mean', 'ndwi_mean', 'kd490', 'chlor_a', 'nflh', 'month_sin', 'month_cos', 'ndwi_std', 'rednir_std', 'hab_label', 'hab_label_heuristic', 'hab_label_final', 'sst', '

## 9.2 Typical run sequence (uncomment)


In [ ]:

# sh("python scripts/fusion/build_fusion_training_table.py --out runs/datasets/fusion_training_with_plants.csv")
# sh("python scripts/fusion/clean_fusion_table.py --in_csv runs/datasets/fusion_training_with_plants.csv --out runs/datasets/fusion_training_with_plants_clean.csv")
# sh("python scripts/fusion/fit_decision_fusion.py --train_csv runs/datasets/fusion_training_with_plants_clean.csv --out_dir runs/eval/fusion/fusion_final2_cv5")
